In [0]:
CREATE OR REFRESH STREAMING TABLE stg_customers
COMMENT "Flattened customer CDC events, ready for APPLY CHANGES INTO"
AS
SELECT
  COALESCE(after.id, before.id)               AS id,
  COALESCE(after.first_name, before.first_name) AS first_name,
  COALESCE(after.last_name, before.last_name)   AS last_name,
  COALESCE(after.email, before.email)           AS email,
  op,
  ts_ms
FROM STREAM(bronze_customers);

In [0]:
CREATE OR REFRESH STREAMING TABLE silver_customers
COMMENT "Customer dimension with full history (SCD Type 2)"
TBLPROPERTIES ("quality" = "silver");
 
APPLY CHANGES INTO silver_customers
FROM STREAM(stg_customers)
KEYS (id)
APPLY AS DELETE WHEN op = "d"
SEQUENCE BY ts_ms
COLUMNS * EXCEPT (op, ts_ms)
STORED AS SCD TYPE 2;

In [0]:
CREATE OR REFRESH STREAMING TABLE stg_products
COMMENT "Flattened product CDC events"
AS
SELECT
  COALESCE(after.id, before.id)                       AS id,
  COALESCE(after.name, before.name)                   AS name,
  COALESCE(after.description, before.description)     AS description,
  COALESCE(after.weight, before.weight)                AS weight,
  op,
  ts_ms
FROM STREAM(bronze_products);

In [0]:
CREATE OR REFRESH STREAMING TABLE silver_products
COMMENT "Product dimension, current state only (SCD Type 1)"
TBLPROPERTIES ("quality" = "silver");
 
APPLY CHANGES INTO silver_products
FROM STREAM(stg_products)
KEYS (id)
APPLY AS DELETE WHEN op = "d"
SEQUENCE BY ts_ms
COLUMNS * EXCEPT (op, ts_ms)
STORED AS SCD TYPE 1;

In [0]:
CREATE OR REFRESH STREAMING TABLE stg_products_on_hand
COMMENT "Flattened inventory CDC events"
AS
SELECT
  COALESCE(after.product_id, before.product_id) AS product_id,
  COALESCE(after.quantity, before.quantity)     AS quantity,
  op,
  ts_ms
FROM STREAM(bronze_products_on_hand);

In [0]:
CREATE OR REFRESH STREAMING TABLE silver_products_on_hand
COMMENT "Current inventory levels (SCD Type 1)"
TBLPROPERTIES ("quality" = "silver");
 
APPLY CHANGES INTO silver_products_on_hand
FROM STREAM(stg_products_on_hand)
KEYS (product_id)
APPLY AS DELETE WHEN op = "d"
SEQUENCE BY ts_ms
COLUMNS * EXCEPT (op, ts_ms)
STORED AS SCD TYPE 1;

In [0]:
CREATE OR REFRESH STREAMING TABLE stg_orders
COMMENT "Flattened order CDC events"
AS
SELECT
  COALESCE(after.id, before.id)               AS id,
  COALESCE(after.order_date, before.order_date) AS order_date,
  COALESCE(after.purchaser, before.purchaser)   AS purchaser,   -- FK -> customers.id
  COALESCE(after.quantity, before.quantity)     AS quantity,
  COALESCE(after.product_id, before.product_id) AS product_id,  -- FK -> products.id
  op,
  ts_ms
FROM STREAM(bronze_orders);

In [0]:
CREATE OR REFRESH STREAMING TABLE silver_orders
COMMENT "Order fact table, current state per order"
TBLPROPERTIES ("quality" = "silver");
 
APPLY CHANGES INTO silver_orders
FROM STREAM(stg_orders)
KEYS (id)
APPLY AS DELETE WHEN op = "d"
SEQUENCE BY ts_ms
COLUMNS * EXCEPT (op, ts_ms)
STORED AS SCD TYPE 1;